In [51]:
!pip install Kanren

In [52]:
from kanren import *
from kanren import lall, Relation, run, eq, facts, var
from kanren.constraints import neq

In [70]:
#facts
parent=Relation()
female=Relation()
male=Relation()
facts(
    parent,
    ("King George VI","Queen Elizabeth II"),
    ("Queen Elizabeth","Queen Elizabeth II"),
    ("King George VI","Margaret"),
    ("Queen Elizabeth","Margaret"),
    ("Queen Elizabeth II","Charles"),
    ("Queen Elizabeth II","Anne"),
    ("Queen Elizabeth II","Prince Andrew"),
    ("Queen Elizabeth II","Prince Edward"),
    ("Prince Philip","Charles"),
    ("Prince Philip","Anne"),
    ("Prince Philip","Prince Andrew"),
    ("Prince Philip","Prince Edward"),
    ("Charles","Prince Williams"),
    ("Charles","Prince Harry"),
    ("Diana","Prince Williams"),
    ("Diana","Prince Harry"),
    ("Prince Williams","Prince George"),
    ("Prince Williams","Princess Charlotte"),
    ("Prince Williams","Prince Louis"),
    ("Catherine","Prince George"),
    ("Catherine","Princess Charlotte"),
    ("Catherine","Prince Louis"),
    ("Prince Harry","Archie Harrison"),
    ("Meghan Markle","Archie Harrison"),
)
facts(
    male,
    ("King George VI",),
    ("Charles",),
    ("Prince Andrew",),
    ("Prince Edward",),
    ("Prince Philip",),
    ("Prince Williams",),
    ("Prince Harry",),
    ("Prince George",),
    ("Prince Louis",),
    ("Archie Harrison",),
)
facts(
    female,
    ("Queen Elizabeth",),
    ("Queen Elizabeth II",),
    ("Margaret",),
    ("Anne",),
    ("Diana",),
    ("Princess Charlotte",),
    ("Catherine",),
    ("Meghan Markle",),
)

In [79]:
def are_siblings(x, y):
    #x and y are siblings if they share at least one parent(p) AND they are not the same person.
    p=var()
    return lall( 
        parent(p,x),
        parent(p,y),
        neq(x,y)
    )


def are_sisters(x, y):
    #x and y are sisters if they are siblings AND both are female
    return lall(
        female(x),
        female(y),
        are_siblings(x,y)
        
    )


def are_brothers(x, y):
    #x and y are brothers if they are siblings and both are male.
    return lall(
        male(x),
        male(y),
        are_siblings(x,y)
    )


def is_mother_of(x, y):
    #x is mother of y if x is parent of y AND x is female
    return lall(
        parent(x,y),
        female(x)
    )


def is_father_of(x, y):
    #x is father of y if x is parent of y AND x is male
    return lall(
        parent(x,y),
        male(x)
    )


def is_grandmother_of(x, y):
    #x is grandmother of y if there's a person(p) that is a child of x AND a parent of y AND x is female
    p=var()
    return lall(
        parent(x,p),
        parent(p,y),
        female(x)
    )


def is_grandfather_of(x, y):
    #x is grandfather of y if there is a person(p) that is a child of x AND parent of y AND x is male
    p=var()
    return lall(
        parent(x,p),
        parent(p,y),
        male(x)
        
    )


def are_cousins(x, y):
    #x and y are cousins if parent of x(px) and parent of y(py) are siblings
    px=var()
    py=var()
    return lall(
        parent(px,x),
        parent(py,y),
        are_siblings(px,py)
    )


In [31]:
kb = lall()
#I left it empty because I already wrote facts inside relation objects, and kanren will automatically find them during unification

In [55]:
# Are A and B cousins?
q=var()
solution = run(0, q, kb, are_cousins("Charles","Diana"))
print(bool(solution))

False


In [81]:
# Are A and B cousins?
solution = run(0, q, kb, are_cousins("Prince George","Archie Harrison"))
print(bool(solution))

True


In [82]:
# Are A and B sisters?
solution=run(0, q, kb, are_sisters("Queen Elizabeth II","Margaret"))
print(bool(solution))

True


In [38]:
#Are A and B siblings?
solution=run(0, q, kb, are_siblings("Princess Charlotte","Prince Louis"))
print(bool(solution))

True


In [40]:
#Are A and B sisters?
solution=run(0,q,kb,are_sisters("Princess Charlotte","Prince Louis"))
print(bool(solution))

False


In [57]:
#List of all siblings of A
solution=run(0, q, kb, are_siblings(q,"Charles"))
print(set(solution))

{'Prince Edward', 'Prince Andrew', 'Anne'}


In [84]:
# Who is the grandmother of B?
solution=run(0,q,kb,is_grandmother_of(q,"Charles"))
print(solution)

('Queen Elizabeth',)


In [85]:
# Who is the grandmother of B?
solution=run(0,q,kb,is_grandmother_of(q,"Prince George"))
print(solution)

('Diana',)


In [86]:
#who is the grandfather of B?
solution=run(0,q,kb,is_grandfather_of(q,"Prince Williams"))
print(solution)

('Prince Philip',)
